# Sprint 2 — StatsBomb 360 Preprocessing for Causal-Emergence Analysis

This notebook turns raw **StatsBomb 360** freeze-frame data into a clean, analysis-ready
dataset for the causal-emergence (Psi) pipeline. It performs, end to end:

1. **Calculate velocities and distances** — 1 Hz tracking reconstruction, finite-difference
   velocities, distance-from-centre and pairwise distances.
2. **Filter goalkeepers and red-card periods** — drop keepers; drop the numerical-disadvantage
   window of any team that receives a red / second-yellow card.
3. **Align tracking with event data** — join freeze frames to events by `event_uuid` to recover
   match time, ball location, possession; tag **goals** and **shots**.
4. **Extract 60-second sliding windows** — build the per-window **micro** (X) and **macro** (V)
   feature matrices consumed by `compute_psi` in `extract_upsets.r`.

Outputs (written to `data/preprocessed/`): a tidy tracking table, an event-alignment table,
and `.npz` feature-matrix bundles per match-team. The logic mirrors
`src/extract_upsets.r` (`reconstruct_tracking`, `match_emergence`).

In [37]:
# === Imports & configuration =================================================
import json, os, glob, urllib.request
import numpy as np
import pandas as pd
import networkx as nx

# --- paths (relative to src/) ---
SB_360_DIR    = os.path.join("..", "statsbomb-360", "data", "three-sixty")
SB_MATCH_DIR  = os.path.join("..", "statsbomb-360", "data", "matches")
SB_EVENTS_DIR = os.path.join("..", "open-data", "data", "events")
GH_EVENTS     = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/{}.json"
OUT_DIR       = os.path.join("data", "preprocessed")
os.makedirs(OUT_DIR, exist_ok=True)

# --- pipeline parameters (match extract_upsets.r) ---
FPS          = 1.0                       # 1 Hz reconstruction (Cheng+ 2025)
WINDOW_SEC   = 60                         # sliding window for Psi
MAX_OUTFIELD = 10                         # outfield players kept per team
PITCH_CENTRE = np.array([60.0, 40.0])     # StatsBomb pitch is 120 x 80
RED_CARDS    = {"Red Card", "Second Yellow"}

print(f"FPS={FPS}  WINDOW_SEC={WINDOW_SEC}  MAX_OUTFIELD={MAX_OUTFIELD}  centre={PITCH_CENTRE}")

FPS=1.0  WINDOW_SEC=60  MAX_OUTFIELD=10  centre=[60. 40.]


## 1. Loaders

Read the 360 freeze frames, the events (local clone first, GitHub raw fallback), and the
match metadata that gives the home/away team names needed for team assignment.

In [38]:
def read_360(match_id):
    """Load 360 freeze-frame records for a match (list of dicts), or None."""
    path = os.path.join(SB_360_DIR, f"{match_id}.json")
    return json.load(open(path)) if os.path.exists(path) else None

def read_events(match_id):
    """Load StatsBomb events; local clone first, then GitHub open-data raw."""
    path = os.path.join(SB_EVENTS_DIR, f"{match_id}.json")
    if os.path.exists(path):
        return json.load(open(path))
    try:
        return json.loads(urllib.request.urlopen(GH_EVENTS.format(match_id), timeout=30).read())
    except Exception as e:
        print(f"  events fetch failed for {match_id}: {e}")
        return None

def match_teams(match_id):
    """Return (home_team, away_team) from the matches metadata, or (None, None)."""
    for path in glob.glob(os.path.join(SB_MATCH_DIR, "*", "*.json")):
        try:
            for m in json.load(open(path)):
                if m.get("match_id") == match_id:
                    return (m["home_team"]["home_team_name"],
                            m["away_team"]["away_team_name"])
        except Exception:
            continue
    return (None, None)

def matches_with_360(limit=None):
    """All match_ids that have a 360 freeze-frame file locally."""
    ids = [int(os.path.splitext(os.path.basename(p))[0])
           for p in glob.glob(os.path.join(SB_360_DIR, "*.json"))]
    ids.sort()
    return ids[:limit] if limit else ids

print(f"{len(matches_with_360())} matches have local 360 data.")

426 matches have local 360 data.


## 2. Event table & red-card periods

Flatten events to one row per event with the fields the pipeline needs, compute an absolute
match clock `t_sec`, and detect red-card sending-off times per team.

In [39]:
def build_events_table(events):
    """Flatten raw events into a tidy DataFrame keyed by event_uuid (= event id)."""
    rows = []
    for e in events:
        loc = e.get("location") or [None, None]
        rows.append(dict(
            event_uuid  = e["id"],
            period      = e["period"],
            minute      = e["minute"],
            second      = e["second"],
            type        = e["type"]["name"],
            team        = (e.get("team") or {}).get("name"),
            poss_team   = (e.get("possession_team") or {}).get("name"),
            ball_x      = loc[0] if loc else None,
            ball_y      = loc[1] if len(loc) > 1 else None,
            shot_outcome= (e.get("shot") or {}).get("outcome", {}).get("name"),
            bb_card     = (e.get("bad_behaviour") or {}).get("card", {}).get("name"),
            foul_card   = (e.get("foul_committed") or {}).get("card", {}).get("name"),
        ))
    ev = pd.DataFrame(rows)
    # absolute clock: 45-min halves; extra time periods continue the count
    ev["t_sec"] = (ev.period - 1) * 45 * 60 + ev.minute * 60 + ev.second
    ev["is_shot"] = ev.type.eq("Shot")
    ev["is_goal"] = ev.shot_outcome.eq("Goal") | ev.type.eq("Own Goal Against")
    return ev

def detect_red_card_periods(ev):
    """Return {team: t_sec_of_sending_off} for each team reduced to 10 men.
    A team is in a 'red-card period' for all t >= that time."""
    sent = ev[(ev.bb_card.isin(RED_CARDS)) | (ev.foul_card.isin(RED_CARDS))]
    out = {}
    for _, r in sent.iterrows():
        tm = r.team
        if tm is not None:
            out[tm] = min(out.get(tm, np.inf), r.t_sec)
    return out

## 3. Freeze frames -> aligned, team-tagged, keeper-filtered players

Each freeze-frame player carries `teammate`, `actor`, `keeper` flags and a pitch location.
Joining on `event_uuid` recovers the timestamp, ball position and possession team. We then:
assign each player to the real team (`teammate` => possession team), **drop goalkeepers**, and
**drop red-card periods** (a sent-off team's frames after its sending-off).

In [40]:
def build_freeze_table(frames):
    """One row per (event, player) from the 360 freeze frames."""
    rows = []
    for f in frames:
        uid = f["event_uuid"]
        for p in f["freeze_frame"]:
            rows.append(dict(event_uuid=uid, teammate=p["teammate"],
                             actor=p["actor"], keeper=p["keeper"],
                             x=p["location"][0], y=p["location"][1]))
    return pd.DataFrame(rows)

def align_and_clean(frames, ev, home, away, red_periods):
    """Join freeze frames to events, assign teams, drop keepers & red-card periods."""
    ft = build_freeze_table(frames)
    cols = ["event_uuid", "t_sec", "poss_team", "ball_x", "ball_y"]
    j = ft.merge(ev[cols], on="event_uuid", how="left").dropna(subset=["t_sec", "x", "y"])

    # team assignment: teammate => possession team, else the other team
    j["team"] = np.where(j.teammate, j.poss_team,
                         np.where(j.poss_team == home, away, home))
    j = j.dropna(subset=["team"])

    # (a) filter goalkeepers
    n0 = len(j); j = j[~j.keeper].copy()
    # (b) filter red-card periods: drop a team's frames at/after its sending-off
    if red_periods:
        keep = np.ones(len(j), dtype=bool)
        for tm, t_red in red_periods.items():
            keep &= ~((j.team.values == tm) & (j.t_sec.values >= t_red))
        n1 = len(j); j = j[keep].copy()
        print(f"  dropped {n0-n1} keeper rows + red-card rows; {len(j)} player-frames left")
    else:
        print(f"  dropped {n0-len(j)} keeper rows; {len(j)} player-frames left (no red cards)")
    return j

## 4. Reconstruct 1 Hz tracking

Freeze frames are event-sampled (irregular). Within each team we order players per frame by
`(x, y)` for a stable index, then linearly interpolate every player onto a 1 Hz grid
(`np.interp`, edge-clamped — equivalent to R's `approx(rule = 2)`), masking times outside a
player's observed range.

In [41]:
def reconstruct_team_tracking(team_df):
    """Irregular event-sampled positions -> regular 1 Hz tracklets for one team.
    Returns tidy DataFrame (player_idx, t, x, y) or None."""
    g = team_df.sort_values(["t_sec", "x", "y"]).copy()
    g["player_idx"] = g.groupby("t_sec").cumcount() + 1          # stable per-frame ordering
    g = g[g.player_idx <= MAX_OUTFIELD]

    grid = np.arange(int(np.floor(g.t_sec.min())),
                     int(np.ceil(g.t_sec.max())) + 1, 1.0 / FPS)
    out = []
    for pid, pg in g.groupby("player_idx"):
        pg = pg.drop_duplicates("t_sec").sort_values("t_sec")
        if len(pg) < 2:
            continue
        xs = np.interp(grid, pg.t_sec, pg.x)
        ys = np.interp(grid, pg.t_sec, pg.y)
        in_range = (grid >= pg.t_sec.min()) & (grid <= pg.t_sec.max())
        xs[~in_range] = np.nan; ys[~in_range] = np.nan
        out.append(pd.DataFrame(dict(player_idx=pid, t=grid, x=xs, y=ys)))
    return pd.concat(out, ignore_index=True) if out else None

## 5. Velocities and distances

On the 1 Hz grid, velocity is the per-player finite difference of position (units: metres per
second). We also record each player's distance from the pitch centre and instantaneous speed.

In [42]:
def add_kinematics(tr):
    """Add finite-difference velocity, speed, and distance-from-centre columns."""
    tr = tr.sort_values(["player_idx", "t"]).copy()
    tr["vx"] = tr.groupby("player_idx").x.diff()
    tr["vy"] = tr.groupby("player_idx").y.diff()
    tr["speed"]       = np.sqrt(tr.vx**2 + tr.vy**2)
    tr["dist_centre"] = np.sqrt((tr.x - PITCH_CENTRE[0])**2 +
                                (tr.y - PITCH_CENTRE[1])**2)
    return tr

## 6. Align tracking with goals and shots

Tag every 1 Hz timestamp with the count of shots/goals occurring in that second, so each
sliding window can be labelled with the attacking events it contains. This links the
collective-movement features to match outcomes for the downstream analysis.

In [43]:
def event_timeline(ev):
    """Per-second shot/goal counts, indexed by integer t_sec."""
    e = ev[ev.is_shot | ev.is_goal].copy()
    e["t_int"] = e.t_sec.round().astype(int)
    g = e.groupby("t_int").agg(n_shots=("is_shot", "sum"),
                               n_goals=("is_goal", "sum")).reset_index()
    return g.rename(columns={"t_int": "t"})

def attach_events_to_tracking(tr, timeline):
    """Left-join per-second shot/goal counts onto the tracking grid (fill 0)."""
    tr = tr.copy(); tr["t_int"] = tr.t.round().astype(int)
    tr = tr.merge(timeline.rename(columns={"t": "t_int"}), on="t_int", how="left")
    tr[["n_shots", "n_goals"]] = tr[["n_shots", "n_goals"]].fillna(0).astype(int)
    return tr.drop(columns="t_int")

## 7. Sixty-second sliding windows & feature matrices

For each team we build a `(T, P, 2)` position array and derive, per timestep, the **micro**
feature matrices `X` (player-level) and the four **macro** feature series `V` used by
`compute_psi`:

| Symbol | Shape | Meaning |
|--------|-------|---------|
| `X_pos`  | `(T, 2P)` | flattened player positions (micro) |
| `X_dist` | `(T, P)`  | each player's distance from centre (micro) |
| `V_com`   | `(T, 2)` | team centre of mass (macro) |
| `V_dist`  | `(T, 1)` | mean distance of outfielders from centre (macro) |
| `V_cvel`  | `(T, 1)` | velocity-weighted global clustering coefficient (macro) |
| `V_cdist` | `(T, 1)` | inverse-distance global clustering coefficient (macro) |

A **60 s sliding window** is any length-`WINDOW_SEC` contiguous slice of these series; there are
`T - WINDOW_SEC + 1` windows per team, each a ready input to `Psi(X, V)`.

In [44]:
def clust_coef(pos, vel=None, mode="velocity"):
    """Weighted global clustering coefficient over a single frame (mirror of R clust_coef).
    velocity:  w_ij = ||v_i|| ||v_j|| (cos d_theta + 1)    inv_dist: w_ij = 1 / d_ij"""
    n = pos.shape[0]
    if n < 2:
        return np.nan
    W = np.zeros((n, n))
    if mode == "velocity":
        if vel is None:
            return np.nan
        speed = np.sqrt((vel**2).sum(1))
        ang   = np.arctan2(vel[:, 1], vel[:, 0])
        for i in range(n):
            for jx in range(n):
                if i == jx or not np.isfinite(speed[i]) or not np.isfinite(speed[jx]):
                    continue
                if not np.isfinite(ang[i]) or not np.isfinite(ang[jx]):
                    continue
                W[i, jx] = speed[i] * speed[jx] * (np.cos(ang[i] - ang[jx]) + 1)
    else:
        for i in range(n):
            for jx in range(n):
                if i == jx or not np.all(np.isfinite(pos[i])) or not np.all(np.isfinite(pos[jx])):
                    continue
                d = np.sqrt(((pos[i] - pos[jx])**2).sum())
                if np.isfinite(d):
                    W[i, jx] = 1.0 / d if d > 1e-9 else 0.0
    G = nx.from_numpy_array(W)
    val = nx.average_clustering(G, weight="weight")
    return val if np.isfinite(val) else np.nan

def build_feature_matrices(tr):
    """Build (T,P,2) arrays and the micro/macro feature matrices for one team."""
    times = np.sort(tr.t.unique())
    P = int(tr.player_idx.max())
    T = len(times)
    tidx = {t: i for i, t in enumerate(times)}
    pos = np.full((T, P, 2), np.nan)
    vel = np.full((T, P, 2), np.nan)
    for r in tr.itertuples(index=False):
        i, p = tidx[r.t], int(r.player_idx) - 1
        pos[i, p] = (r.x, r.y)
        vel[i, p] = (r.vx, r.vy)

    # micro feature matrices
    X_pos  = pos.reshape(T, P * 2)
    X_dist = np.sqrt(((pos - PITCH_CENTRE)**2).sum(2))           # (T, P)

    # macro feature series
    V_com  = np.nanmean(pos, axis=1)                              # (T, 2)
    V_dist = np.nanmean(np.sqrt(((pos - PITCH_CENTRE)**2).sum(2)),
                        axis=1, keepdims=True)                    # (T, 1)
    V_cvel  = np.full((T, 1), np.nan)
    V_cdist = np.full((T, 1), np.nan)
    for k in range(T):
        ok = np.all(np.isfinite(pos[k]), axis=1)
        if ok.sum() >= 2:
            V_cvel[k, 0]  = clust_coef(pos[k, ok], vel[k, ok], "velocity")
            V_cdist[k, 0] = clust_coef(pos[k, ok], mode="inv_dist")

    return dict(times=times, n_players=P, pos=pos, vel=vel,
                X_pos=X_pos, X_dist=X_dist,
                V_com=V_com, V_dist=V_dist, V_cvel=V_cvel, V_cdist=V_cdist)

def sliding_windows(T, window=WINDOW_SEC):
    """Yield (start, stop) index pairs for every contiguous length-`window` slice."""
    return [(s, s + window) for s in range(max(0, T - window + 1))]

## 8. Full per-match pipeline

`preprocess_match` chains every step and returns (1) the **processed dataset** (tidy 1 Hz
tracking with kinematics + event tags for both teams) and (2) the **feature matrices** keyed by
team. This is the object the causal-emergence analysis consumes.

In [45]:
def preprocess_match(match_id, verbose=True):
    """Clean one match end to end. Returns dict(tracking=DataFrame, features={team:dict}) or None."""
    frames = read_360(match_id)
    events = read_events(match_id)
    home, away = match_teams(match_id)
    if not frames or not events or home is None:
        if verbose: print(f"[{match_id}] missing 360/events/metadata -> skip")
        return None
    if verbose: print(f"[{match_id}] {home} vs {away}")

    ev = build_events_table(events)
    red_periods = detect_red_card_periods(ev)
    if verbose and red_periods:
        print(f"  red-card periods: {red_periods}")

    players = align_and_clean(frames, ev, home, away, red_periods)   # steps 2-3
    timeline = event_timeline(ev)                                    # step 6 source

    tracking_parts, features = [], {}
    for tm in players.team.dropna().unique():
        tr = reconstruct_team_tracking(players[players.team == tm])  # step 4
        if tr is None:
            continue
        tr = add_kinematics(tr)                                      # step 1 (vel/dist)
        tr = attach_events_to_tracking(tr, timeline)                 # step 6 (goals/shots)
        tr["team"] = tm; tr["match_id"] = match_id
        tracking_parts.append(tr)
        features[tm] = build_feature_matrices(tr)                    # step 7
        if verbose:
            nwin = len(sliding_windows(len(features[tm]["times"])))
            print(f"  {tm}: T={len(features[tm]['times'])}s, P={features[tm]['n_players']}, "
                  f"{nwin} sliding 60s windows, mean speed {tr.speed.mean():.2f} m/s")

    if not tracking_parts:
        return None
    tracking = pd.concat(tracking_parts, ignore_index=True)
    return dict(match_id=match_id, home=home, away=away,
                tracking=tracking, features=features)

## 9. Run on an example match and save

Run the pipeline on one 360 match, inspect the **processed dataset** and a **feature matrix**,
then persist outputs to `data/preprocessed/` (tracking as Parquet; feature matrices as `.npz`).
Loop over `matches_with_360()` to batch the full set.

In [46]:
def save_match(result, out_dir=OUT_DIR):
    """Persist tracking (Parquet) and per-team feature matrices (.npz)."""
    mid = result["match_id"]
    result["tracking"].to_parquet(os.path.join(out_dir, f"{mid}_tracking.parquet"), index=False)
    for tm, F in result["features"].items():
        safe = "".join(c if c.isalnum() else "_" for c in tm)
        np.savez_compressed(
            os.path.join(out_dir, f"{mid}_{safe}_features.npz"),
            times=F["times"], pos=F["pos"], vel=F["vel"],
            X_pos=F["X_pos"], X_dist=F["X_dist"],
            V_com=F["V_com"], V_dist=F["V_dist"],
            V_cvel=F["V_cvel"], V_cdist=F["V_cdist"])
    return mid

# --- demo on the first available 360 match ---
example_id = matches_with_360(limit=1)[0]
result = preprocess_match(example_id)

if result is not None:
    print("\nProcessed dataset (head):")
    display(result["tracking"].head())
    tm0 = next(iter(result["features"]))
    F = result["features"][tm0]
    print(f"\nFeature matrices for {tm0}:")
    for k in ["X_pos", "X_dist", "V_com", "V_dist", "V_cvel", "V_cdist"]:
        print(f"  {k:8s} shape {F[k].shape}")
    print(f"  -> {len(sliding_windows(len(F['times'])))} sliding 60s windows ready for Psi")
    save_match(result)
    print(f"\nSaved outputs to {OUT_DIR}/")

[3764440] Barcelona vs Elche
  dropped 1419 keeper rows; 67465 player-frames left (no red cards)
  Barcelona: T=8279s, P=10, 8220 sliding 60s windows, mean speed 7.35 m/s
  Elche: T=8279s, P=10, 8220 sliding 60s windows, mean speed 6.80 m/s

Processed dataset (head):


,player_idx,t,x,y,vx,vy,speed,dist_centre,n_shots,n_goals,team,match_id
0,1,0.0,52.273950,38.428478,NaN,NaN,NaN,7.884258,0,0,Barcelona,3764440
1,1,1.0,58.519454,31.526446,6.245504,-6.902031,9.308295,8.601926,0,0,Barcelona,3764440
2,1,2.0,56.949172,39.977887,-1.570283,8.451441,8.596083,3.050908,0,0,Barcelona,3764440
3,1,3.0,55.378889,48.429328,-1.570283,8.451441,8.596083,9.612920,0,0,Barcelona,3764440
4,1,4.0,53.117556,32.270825,-2.261334,-16.158502,16.315968,10.349308,0,0,Barcelona,3764440



Feature matrices for Barcelona:
  X_pos    shape (8279, 20)
  X_dist   shape (8279, 10)
  V_com    shape (8279, 2)
  V_dist   shape (8279, 1)
  V_cvel   shape (8279, 1)
  V_cdist  shape (8279, 1)
  -> 8220 sliding 60s windows ready for Psi

Saved outputs to data/preprocessed/



Saved outputs to data/preprocessed/


### 9.1  Before vs. after preprocessing

Side-by-side look at the **same match** before and after the pipeline. *Before* is the
raw StatsBomb input: event-sampled freeze frames keyed by `event_uuid`, with goalkeepers
present, no match clock, no team label, and no kinematics — plus the separate events table.
*After* is the tidy 1 Hz tracking table: keepers removed, team resolved, regular time grid,
finite-difference velocities/distances, and per-second shot/goal tags.

In [47]:
# --- BEFORE: raw inputs (unchanged source data) ---
raw_frames = read_360(example_id)
raw_events = read_events(example_id)
raw_ff     = build_freeze_table(raw_frames)        # one row per (event, player)
raw_ev     = build_events_table(raw_events)        # one row per event

print(f"BEFORE  match {example_id}")
print(f"  freeze-frame player-rows : {len(raw_ff):,}  (includes goalkeepers, no time/team/velocity)")
print(f"  keeper rows present      : {int(raw_ff.keeper.sum()):,}")
print(f"  raw events               : {len(raw_ev):,}")
print("\n  raw freeze-frame rows (note: keeper flag, no t / team / velocity):")
display(raw_ff.head(6))
print("  raw events (event-sampled, irregular timing):")
display(raw_ev[["event_uuid","period","minute","second","t_sec","type","team","poss_team","ball_x","ball_y"]].head(6))

# --- AFTER: processed tracking table from the demo run ---
proc = result["tracking"]
print(f"\nAFTER   match {example_id}")
print(f"  processed player-seconds : {len(proc):,}  (1 Hz, keepers removed, kinematics + event tags)")
print(f"  keeper rows present      : 0  (filtered)")
print(f"  teams / players each     : {proc.team.unique().tolist()} / "
      f"{proc.groupby('team').player_idx.nunique().to_dict()}")
print("\n  processed tracking rows:")
display(proc.head(6))

BEFORE  match 3764440
  freeze-frame player-rows : 68,884  (includes goalkeepers, no time/team/velocity)
  keeper rows present      : 1,419
  raw events               : 4,160

  raw freeze-frame rows (note: keeper flag, no t / team / velocity):


,event_uuid,teammate,actor,keeper,x,y
0,23743e50-fbe4-4929-938e-9b3ab39ff4ff,False,False,False,52.273950,38.428478
1,23743e50-fbe4-4929-938e-9b3ab39ff4ff,False,False,False,54.327138,47.166411
2,23743e50-fbe4-4929-938e-9b3ab39ff4ff,False,False,False,55.867030,31.488913
3,23743e50-fbe4-4929-938e-9b3ab39ff4ff,False,False,False,58.946812,55.132085
4,23743e50-fbe4-4929-938e-9b3ab39ff4ff,True,True,False,61.000000,40.099998
5,23743e50-fbe4-4929-938e-9b3ab39ff4ff,True,False,False,61.256649,56.417276


  raw events (event-sampled, irregular timing):


,event_uuid,period,minute,second,t_sec,type,team,poss_team,ball_x,ball_y
0,d130c53e-c291-48bc-8372-721330fb160b,1,0,0,0,Starting XI,Barcelona,Barcelona,NaN,NaN
1,d5c25be3-487e-40a0-887d-f5d274d2d57b,1,0,0,0,Starting XI,Elche,Barcelona,NaN,NaN
2,900ca195-c93a-4143-874e-ce2ad56c3752,1,0,0,0,Half Start,Barcelona,Barcelona,NaN,NaN
3,0ce64a65-8a8b-479b-bdc4-eaec24d8423d,1,0,0,0,Half Start,Elche,Barcelona,NaN,NaN
4,23743e50-fbe4-4929-938e-9b3ab39ff4ff,1,0,0,0,Pass,Elche,Elche,61.0,40.1
5,7b823cb0-f2d0-49fb-b7ba-1c2941fd8752,1,0,1,1,Ball Receipt*,Elche,Elche,49.2,50.3



AFTER   match 3764440
  processed player-seconds : 165,580  (1 Hz, keepers removed, kinematics + event tags)
  keeper rows present      : 0  (filtered)
  teams / players each     : ['Barcelona', 'Elche'] / {'Barcelona': 10, 'Elche': 10}

  processed tracking rows:


,player_idx,t,x,y,vx,vy,speed,dist_centre,n_shots,n_goals,team,match_id
0,1,0.0,52.273950,38.428478,NaN,NaN,NaN,7.884258,0,0,Barcelona,3764440
1,1,1.0,58.519454,31.526446,6.245504,-6.902031,9.308295,8.601926,0,0,Barcelona,3764440
2,1,2.0,56.949172,39.977887,-1.570283,8.451441,8.596083,3.050908,0,0,Barcelona,3764440
3,1,3.0,55.378889,48.429328,-1.570283,8.451441,8.596083,9.612920,0,0,Barcelona,3764440
4,1,4.0,53.117556,32.270825,-2.261334,-16.158502,16.315968,10.349308,0,0,Barcelona,3764440
5,1,5.0,50.257032,30.304404,-2.860523,-1.966421,3.471225,13.745181,0,0,Barcelona,3764440


### 9.2  Snapshot at a goal — Elche vs. Barcelona

Inspect the filtered, processed rows at the exact second a goal is scored. The code finds
goals from the event table and prefers an **Elche** goal; if Elche did not score in this match
it says so and falls back to the goal that actually occurred. For the demo match
(`3764440`, **Barcelona 3–0 Elche**, La Liga 2020/21) Elche did not score, so the snapshot is
taken at the match's first goal. Every player's position, velocity, distance-from-centre and
the `n_goals` tag are shown for **both** teams at that instant.

In [48]:
# locate goals in the demo match and choose the snapshot second
ev_demo  = build_events_table(read_events(example_id))
goals    = ev_demo[ev_demo.is_goal].copy()
goals["t_int"] = goals.t_sec.round().astype(int)
elche_goals = goals[goals.team == "Elche"]

if len(elche_goals):
    g = elche_goals.iloc[0]
    print(f"Elche goal found at P{g.period} {int(g.minute):02d}:{int(g.second):02d}  (t={g.t_int}s)")
else:
    g = goals.iloc[0]
    print(f"Note: Elche did NOT score in match {example_id} "
          f"({result['home']} vs {result['away']}; this fixture finished Barcelona 3-0 Elche).")
    print(f"      Showing the match's first goal instead: {g.team} at "
          f"P{g.period} {int(g.minute):02d}:{int(g.second):02d}  (t={g.t_int}s).")

t_goal = float(g.t_int)
snap = (result["tracking"][result["tracking"].t == t_goal]
        .sort_values(["team", "player_idx"]))
cols = ["match_id","team","player_idx","t","x","y","vx","vy","speed","dist_centre","n_shots","n_goals"]
print(f"\nFiltered tracking at the goal second t={t_goal:.0f}  "
      f"({len(snap)} rows = {snap.team.nunique()} teams x outfielders):")
display(snap[cols])

# short pre-goal evolution for the attacking side (a few seconds up to the goal)
atk = g.team if g.team in result["tracking"].team.unique() else result["tracking"].team.unique()[0]
ctx = (result["tracking"]
       .query("team == @atk and t >= @t_goal - 3 and t <= @t_goal")
       .sort_values(["t","player_idx"]))
print(f"\n{atk} — seconds leading into the goal (t={t_goal-3:.0f}..{t_goal:.0f}), "
      f"team centroid drifting into the box:")
display(ctx.groupby("t")[["x","y","speed","dist_centre","n_shots","n_goals"]].mean().round(2))

Note: Elche did NOT score in match 3764440 (Barcelona vs Elche; this fixture finished Barcelona 3-0 Elche).
      Showing the match's first goal instead: Barcelona at P2 47:05  (t=5525s).

Filtered tracking at the goal second t=5525  (20 rows = 2 teams x outfielders):


,match_id,team,player_idx,t,x,y,vx,vy,speed,dist_centre,n_shots,n_goals
5525,3764440,Barcelona,1,5525.0,85.001521,22.535823,73.424930,-7.240255,73.781038,30.497107,1,1
13804,3764440,Barcelona,2,5525.0,95.794390,38.698996,82.554639,-4.654325,82.685738,35.818026,1,1
22083,3764440,Barcelona,3,5525.0,101.575366,21.711112,88.175366,-12.488888,89.055418,45.420199,1,1
30362,3764440,Barcelona,4,5525.0,104.244028,44.714803,87.860088,-5.795095,88.050997,44.494532,1,1
38641,3764440,Barcelona,5,5525.0,109.396590,53.478283,92.247610,19.282046,94.241280,51.202414,1,1
46920,3764440,Barcelona,6,5525.0,110.800003,42.000000,93.520523,14.213090,94.594398,50.839358,1,1
55199,3764440,Barcelona,7,5525.0,16.389579,31.631047,-1.156807,1.111265,1.604092,44.406173,1,1
63478,3764440,Barcelona,8,5525.0,29.996076,30.814387,0.421428,-0.166867,0.453261,31.378511,1,1
71757,3764440,Barcelona,9,5525.0,33.397525,41.877515,0.479974,0.171075,0.509551,26.668647,1,1
80036,3764440,Barcelona,10,5525.0,80.088008,23.068141,-0.630600,0.625292,0.888058,26.271960,1,1



Barcelona — seconds leading into the goal (t=5522..5525), team centroid drifting into the box:


,x,y,speed,dist_centre,n_shots,n_goals
t,,,,,,
5522.0,78.20,47.56,9.84,24.98,0.0,0.0
5523.0,82.78,45.39,9.84,30.79,0.0,0.0
5524.0,24.98,34.55,62.30,40.54,0.0,0.0
5525.0,76.67,35.05,52.59,38.70,1.0,1.0


In [49]:
# === Batch over all local 360 matches (uncomment to run the full set) ========
# done, failed = [], []
# for mid in matches_with_360():
#     try:
#         r = preprocess_match(mid, verbose=False)
#         if r is not None:
#             save_match(r); done.append(mid)
#         else:
#             failed.append(mid)
#     except Exception as e:
#         print(f"[{mid}] error: {e}"); failed.append(mid)
# print(f"preprocessed {len(done)} matches; {len(failed)} skipped/failed")